In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Accidents Dataset") \
    .getOrCreate()

file_path = "hdfs://nameNode:9000/user/gyembo/project/US_Accidents_March23.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)

df.show(5)
df.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/22 09:58:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/22 09:58:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+-------+--------+-------------------+-------------------+-----------------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID| Source|Severity|         Start_Time|           End_Time|        Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Ameni

In [2]:
df.summary().show(vertical=True)


-RECORD 0-------------------------------------
 summary               | count                
 ID                    | 7728394              
 Source                | 7728394              
 Severity              | 7728394              
 Start_Lat             | 7728394              
 Start_Lng             | 7728394              
 End_Lat               | 4325632              
 End_Lng               | 4325632              
 Distance(mi)          | 7728394              
 Description           | 7728389              
 Street                | 7717525              
 City                  | 7728141              
 County                | 7728394              
 State                 | 7728394              
 Zipcode               | 7726479              
 Country               | 7728394              
 Timezone              | 7720586              
 Airport_Code          | 7705759              
 Temperature(F)        | 7564541              
 Wind_Chill(F)         | 5729375              
 Humidity(%) 

In [3]:
df.columns

['ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'End_Lat',
 'End_Lng',
 'Distance(mi)',
 'Description',
 'Street',
 'City',
 'County',
 'State',
 'Zipcode',
 'Country',
 'Timezone',
 'Airport_Code',
 'Weather_Timestamp',
 'Temperature(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Direction',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [4]:
# Columns to remove
columns_to_drop = [
    'ID',
    'Source',
    'End_Lat',
    'End_Lng',
    'Description',
    'Street',
    'Zipcode',
    'Airport_Code',
    'Timezone',
    'Bump',
    'Give_Way',
    'No_Exit',
    'Roundabout',
    'Turning_Loop',
    'Traffic_Calming',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight'
]

# Drop irrelevant columns
df = df.drop(*columns_to_drop)

# Show remaining columns
print(df.columns)

# Display first 5 rows
df.show(5)

['Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'City', 'County', 'State', 'Country', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
+--------+-------------------+-------------------+-----------------+------------------+------------+------------+----------+-----+-------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+--------+--------+-------+-------+-----+--------------+--------------+
|Severity|         Start_Time|           End_Time|        Start_Lat|         Start_Lng|Distance(mi)|        City|    County|State|Country|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(

In [5]:
from pyspark.sql.functions import col, when, count, isnan
from pyspark.sql.types import DoubleType, FloatType

exprs = []

for c, dtype in df.dtypes:
    
    if dtype in ['double', 'float', 'int', 'bigint']:
        exprs.append(
            count(
                when(col(c).isNull() | isnan(col(c)), c)
            ).alias(c)
        )
    else:
        exprs.append(
            count(
                when(col(c).isNull(), c)
            ).alias(c)
        )

null_counts = df.select(exprs)

null_counts.show(vertical=True)

-RECORD 0--------------------
 Severity          | 0       
 Start_Time        | 0       
 End_Time          | 0       
 Start_Lat         | 0       
 Start_Lng         | 0       
 Distance(mi)      | 0       
 City              | 253     
 County            | 0       
 State             | 0       
 Country           | 0       
 Weather_Timestamp | 120228  
 Temperature(F)    | 163853  
 Wind_Chill(F)     | 1999019 
 Humidity(%)       | 174144  
 Pressure(in)      | 140679  
 Visibility(mi)    | 177098  
 Wind_Direction    | 175206  
 Wind_Speed(mph)   | 571233  
 Precipitation(in) | 2203586 
 Weather_Condition | 173459  
 Amenity           | 0       
 Crossing          | 0       
 Junction          | 0       
 Railway           | 0       
 Station           | 0       
 Stop              | 0       
 Traffic_Signal    | 0       
 Sunrise_Sunset    | 23246   



In [10]:
df = df.drop(
    'Wind_Chill(F)',
    'Precipitation(in)',
    'Weather_Timestamp'
)

In [11]:
from pyspark.sql.functions import col, when

df = df.fillna({
    "Temperature(F)": df.approxQuantile("Temperature(F)", [0.5], 0.25)[0],
    "Humidity(%)": df.approxQuantile("Humidity(%)", [0.5], 0.25)[0],
    "Pressure(in)": df.approxQuantile("Pressure(in)", [0.5], 0.25)[0],
    "Visibility(mi)": df.approxQuantile("Visibility(mi)", [0.5], 0.25)[0],
    "Wind_Speed(mph)": df.approxQuantile("Wind_Speed(mph)", [0.5], 0.25)[0]
})

In [13]:
df = df.fillna({
    "City": "Unknown",
    "Wind_Direction": "Unknown",
    "Weather_Condition": "Unknown",
    "Sunrise_Sunset": "Unknown"
})

In [15]:
exprs = []

for c, dtype in df.dtypes:
    
    if dtype in ['double', 'float', 'int', 'bigint']:
        exprs.append(
            count(
                when(col(c).isNull() | isnan(col(c)), c)
            ).alias(c)
        )
    else:
        exprs.append(
            count(
                when(col(c).isNull(), c)
            ).alias(c)
        )

null_counts = df.select(exprs)

null_counts.show(vertical=True)

-RECORD 0----------------
 Severity          | 0   
 Start_Time        | 0   
 End_Time          | 0   
 Start_Lat         | 0   
 Start_Lng         | 0   
 Distance(mi)      | 0   
 City              | 0   
 County            | 0   
 State             | 0   
 Country           | 0   
 Temperature(F)    | 0   
 Humidity(%)       | 0   
 Pressure(in)      | 0   
 Visibility(mi)    | 0   
 Wind_Direction    | 0   
 Wind_Speed(mph)   | 0   
 Weather_Condition | 0   
 Amenity           | 0   
 Crossing          | 0   
 Junction          | 0   
 Railway           | 0   
 Station           | 0   
 Stop              | 0   
 Traffic_Signal    | 0   
 Sunrise_Sunset    | 0   



In [16]:
df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- City: string (nullable = false)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Temperature(F): double (nullable = false)
 |-- Humidity(%): double (nullable = false)
 |-- Pressure(in): double (nullable = false)
 |-- Visibility(mi): double (nullable = false)
 |-- Wind_Direction: string (nullable = false)
 |-- Wind_Speed(mph): double (nullable = false)
 |-- Weather_Condition: string (nullable = false)
 |-- Amenity: boolean (nullable = true)
 |-- Crossing: boolean (nullable = true)
 |-- Junction: boolean (nullable = true)
 |-- Railway: boolean (nullable = true)
 |-- Station: boolean (nullable = true)
 |-- Stop: boolean (nullable = true)
 |-- Traffic_Signal: b

In [18]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col

df = df.withColumn(
    "Severity",
    col("Severity").cast(IntegerType())
)

In [19]:
from pyspark.sql.types import DoubleType

numeric_cols = [
    "Start_Lat",
    "Start_Lng",
    "Distance(mi)",
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)"
]

for c in numeric_cols:
    df = df.withColumn(
        c,
        col(c).cast(DoubleType())
    )

In [20]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import lower

string_cols = [
    "City",
    "County",
    "State",
    "Country",
    "Wind_Direction",
    "Weather_Condition",
    "Sunrise_Sunset"
]

for c in string_cols:
    df = df.withColumn(
        c,
        lower(col(c).cast(StringType()))
    )

In [21]:
from pyspark.sql.types import BooleanType

boolean_cols = [
    "Amenity",
    "Crossing",
    "Junction",
    "Railway",
    "Station",
    "Stop",
    "Traffic_Signal"
]

for c in boolean_cols:
    df = df.withColumn(
        c,
        col(c).cast(BooleanType())
    )

In [22]:
df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- City: string (nullable = false)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Temperature(F): double (nullable = false)
 |-- Humidity(%): double (nullable = false)
 |-- Pressure(in): double (nullable = false)
 |-- Visibility(mi): double (nullable = false)
 |-- Wind_Direction: string (nullable = false)
 |-- Wind_Speed(mph): double (nullable = false)
 |-- Weather_Condition: string (nullable = false)
 |-- Amenity: boolean (nullable = true)
 |-- Crossing: boolean (nullable = true)
 |-- Junction: boolean (nullable = true)
 |-- Railway: boolean (nullable = true)
 |-- Station: boolean (nullable = true)
 |-- Stop: boolean (nullable = true)
 |-- Traffic_Signal: b

In [23]:
df = df.filter(df["Distance(mi)"] >= 0)
df = df.filter(df["Visibility(mi)"] >= 0)
df = df.filter(df["Wind_Speed(mph)"] >= 0)

In [25]:
from pyspark.sql.functions import hour, month, dayofweek, year

df = df.withColumn("Hour", hour("Start_Time"))
df = df.withColumn("Month", month("Start_Time"))
df = df.withColumn("DayOfWeek", dayofweek("Start_Time"))
df = df.withColumn("Year", year("Start_Time"))

In [26]:
from pyspark.sql.functions import unix_timestamp

df = df.withColumn(
    "Duration_Minutes",
    (unix_timestamp("End_Time") - unix_timestamp("Start_Time")) / 60
)

In [27]:
df.select(
    "Start_Time",
    "Hour",
    "Month",
    "DayOfWeek",
    "Duration_Minutes"
).show(5)

+-------------------+----+-----+---------+----------------+
|         Start_Time|Hour|Month|DayOfWeek|Duration_Minutes|
+-------------------+----+-----+---------+----------------+
|2016-02-08 05:46:00|   5|    2|        2|           314.0|
|2016-02-08 06:07:59|   6|    2|        2|            30.0|
|2016-02-08 06:49:27|   6|    2|        2|            30.0|
|2016-02-08 07:23:34|   7|    2|        2|            30.0|
|2016-02-08 07:39:07|   7|    2|        2|            30.0|
+-------------------+----+-----+---------+----------------+
only showing top 5 rows



In [34]:
df.write.mode("overwrite").option("header", True).csv(
    "hdfs://namenode:9000/user/gyembo/project/traffic_cleaned_eda_csv"
)